# Hafta 9 — Destek Vektör Makineleri (SVM)

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

İki takımın oyuncuları sahanın iki ayrı bölgesinde dağınık biçimde dursun. Aralarına bir sınır çizgisi çekmeniz gereksin. Bu işi yapabilecek pek çok çizgi vardır. En güvenlisi, iki takımın oyuncularına da en uzak olan, yani aralarında en geniş boşluğu bırakan çizgidir. Destek vektör makineleri bu çizgiyi bulur. Gruplar düz bir çizgiyle ayrılamıyorsa veriyi daha yüksek bir boyuta taşıyarak ayrılabilir hale getirir.

---

## Bu Haftanın Hedefleri

- Maksimum marj fikrini ve destek vektörlerini anlamak
- C parametresinin etkisini görmek
- Kernel hilesinin nasıl çalıştığını sayılarla doğrulamak
- Pipeline ve GridSearchCV ile hiperparametre aramak

---

## 1. Maksimum Marj

İki sınıfı ayıran sonsuz sayıda doğru çizilebilir. SVM, iki sınıfa da **en uzak** olan doğruyu seçer:

```
  ○  ○        ┆    ╱    ┆
    ○   ⊙ ─ ─ ┆ ─ ╱ ─ ─ ┆        ⊙, ⊛ : destek vektörleri
  ○           ┆  ╱      ┆
              ┆ ╱  ⊛    ┆  ●  ●
              ┆╱        ┆    ●
              ←─ marj ──→
```

- **Karar sınırı:** w·x + b = 0
- **Marj:** Sınırın iki yanındaki boşluk; genişliği 2 / ‖w‖
- **Destek vektörleri:** Marjın tam kenarındaki noktalar. Sınırın yerini **yalnızca bunlar** belirler. Diğer noktaları silseniz bile sınır değişmez.

### Elle örnek

Noktalar: (1, 1) → sınıf −1, (3, 3) → sınıf +1, (4, 1) → sınıf +1. Üç nokta da marjın kenarındadır, yani w·x + b değeri pozitifler için +1, negatif için −1 olmalıdır:

```
(1,1):  w₁ + w₂ + b   = −1
(3,3):  3w₁ + 3w₂ + b = +1
(4,1):  4w₁ + w₂ + b  = +1
```

Çözüm: **w = (2/3, 1/3), b = −2**. Marj = 2 / ‖w‖ = 2 / 0.745 ≈ **2.68**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

X = np.array([[1, 1], [3, 3], [4, 1]])
y = np.array([-1, 1, 1])
svm = SVC(kernel="linear", C=1e6).fit(X, y)       # çok büyük C ≈ hiç hata kabul etme
print("w =", svm.coef_, " b =", svm.intercept_)   # w ≈ [[0.667 0.333]], b ≈ [-2.]
print("Marj:", 2 / np.linalg.norm(svm.coef_))

---

## 2. Yumuşak Marj ve C Parametresi

Gerçek verilerde sınıflar genellikle iç içe geçer ve tek bir aykırı nokta marjı çok daraltabilir. **Yumuşak marj**, bazı noktaların marjın içine, hatta yanlış tarafa düşmesine izin verir ama bunu cezalandırır.

**C**, bu cezanın şiddetini belirler:

| C | Davranış | Risk |
|---|---|---|
| Büyük | Her noktayı doğru sınıflandırmaya çalışır, marj dar | Aşırı öğrenme |
| Küçük | Hatalara tolerans gösterir, marj geniş | Eksik öğrenme |

In [ ]:
from sklearn.datasets import make_blobs

def sinir_ciz(model, X, y, ax, baslik):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k")
    ax.set_title(baslik)

X, y = make_blobs(n_samples=100, centers=2, cluster_std=1.6, random_state=4)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, C in zip(ax, [0.01, 1, 100]):
    m = SVC(kernel="linear", C=C).fit(X, y)
    sinir_ciz(m, X, y, a, f"C = {C}  (destek vektörü: {len(m.support_)})")
plt.show()

Kesikli çizgiler marjın kenarlarını gösterir. C küçüldükçe marjın genişlediğine dikkat edin.

---

## 3. Kernel Hilesi

### Sorun

```
1 boyut:   ● ● ○ ○ ○ ○ ● ●       Tek bir eşikle ayrılamaz
          −3 −2 −1 0 1 2 3
```

**Çözüm:** Veriye x² gibi yeni bir boyut eklenir. (x, x²) düzleminde ortadaki ○ noktaları aşağıda, kenardaki ● noktaları yukarıda kalır ve artık düz bir çizgiyle ayrılabilirler.

### Hile

Veriyi gerçekten yüksek boyuta taşımak pahalıdır. SVM'in ise yalnızca noktalar arasındaki **iç çarpımlara** ihtiyacı vardır. **Kernel fonksiyonu**, yüksek boyuttaki iç çarpımı dönüşümü hiç yapmadan doğrudan hesaplar.

**Sayılarla doğrulama:** φ(x) = (x₁², √2·x₁x₂, x₂²) dönüşümü ve x = (1, 2), z = (3, 4) için:

```
Yol 1 — Önce dönüştür, sonra iç çarpım al (3 boyutta):
  φ(x) = (1, 2.83, 4),  φ(z) = (9, 16.97, 16)  →  9 + 48 + 64 = 121

Yol 2 — Kernel ile (2 boyutta kalarak):
  K(x, z) = (x · z)² = (3 + 8)² = 121  ✓
```

In [ ]:
x, z = np.array([1, 2]), np.array([3, 4])
phi = lambda v: np.array([v[0]**2, np.sqrt(2) * v[0] * v[1], v[1]**2])
print(phi(x) @ phi(z), (x @ z) ** 2)        # 121.0 121

### Yaygın kerneller

| Kernel | Ne zaman? |
|---|---|
| Doğrusal | Çok öznitelikli veriler (ör. metin) |
| Polinom | Öznitelik etkileşimleri önemliyse |
| **RBF (Gauss)** | Varsayılan seçim; benzerliği mesafeye göre ölçer |

RBF kernelindeki **γ** parametresi her noktanın etki alanını belirler. γ büyükse her nokta yalnızca yakın çevresini etkiler ve sınır çok girintili olur (aşırı öğrenme). γ küçükse sınır yumuşak ve neredeyse düz olur.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(200, noise=0.1, factor=0.4, random_state=0)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (kernel, kw) in zip(ax, [("linear", {}), ("poly", {"degree": 2}), ("rbf", {"gamma": 1})]):
    m = SVC(kernel=kernel, **kw).fit(X, y)
    sinir_ciz(m, X, y, a, f"{kernel}  (doğruluk {m.score(X, y):.2f})")
plt.show()

---

## 4. Pipeline ve GridSearchCV

SVM mesafeye dayandığı için **ölçekleme şarttır**. Ölçekleme ve modeli Pipeline içinde birleştirirsek, çapraz doğrulamanın her katında ölçekleyici yalnızca o katın eğitim kısmına uyarlanır ve sızıntı olmaz.

**GridSearchCV**, verilen bütün parametre kombinasyonlarını çapraz doğrulamayla dener ve en iyisini seçer.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

pipe = Pipeline([("olcek", StandardScaler()), ("svm", SVC())])
izgara = {"svm__kernel": ["rbf"], "svm__C": [0.1, 1, 10, 100], "svm__gamma": [0.001, 0.01, 0.1, 1]}
arama = GridSearchCV(pipe, izgara, cv=5, n_jobs=-1).fit(X_tr, y_tr)

print("En iyi parametreler:", arama.best_params_)
print("CV doğruluğu       :", round(arama.best_score_, 4))
print("Test doğruluğu     :", round(arama.score(X_te, y_te), 4))

Bu arama 16 kombinasyon × 5 kat = 80 model eğitir.

---

## 5. Laboratuvar

1. `make_moons(200, noise=0.25)` verisinde C ∈ {0.1, 1, 100} ve γ ∈ {0.1, 1, 10} için 3×3'lük bir karar sınırı ızgarası çizin. Hangi kombinasyonlar aşırı, hangileri eksik öğreniyor?
2. Bölüm 4'teki aramaya doğrusal kerneli de ekleyin (`{"svm__kernel": ["linear"], "svm__C": [...]}`) ve sonuçları karşılaştırın.

---

## 6. Özet

- SVM, sınıflar arasındaki marjı en büyük yapan sınırı bulur; sınırı yalnızca destek vektörleri belirler.
- C büyükse marj dar, küçükse geniştir.
- Kernel hilesi, dönüşümü yapmadan yüksek boyutta çalışmayı sağlar. RBF kernelinde γ sınırın esnekliğini belirler.
- Ölçekleme ve hiperparametre araması Pipeline + GridSearchCV ile sızıntısız yapılır.

## Kendinizi Sınayın

1. w = (3, 4) ise marj genişliği nedir?
2. Destek vektörü olmayan bir noktayı veriden çıkarırsak sınır değişir mi?
3. Eğitim doğruluğu %100, CV doğruluğu %70 olan bir RBF SVM'de C ve γ'yı nasıl değiştirirsiniz?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.